### Load, check, and process the Synthetic data of Tech Related Job description

In [163]:
# import necessary libraries
from pathlib import Path

import pandas as pd

# setting for inspecting long text strings
pd.set_option("display.max_colwidth", 200)

In [164]:
data_path = Path("raw/JobDescriptions.csv")
df = pd.read_csv(data_path)

In [165]:
df.head(5)

,JobID,Title,Category,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,PreferredSkills,Education,Location
0,1,Python Developer,INFORMATION-TECHNOLOGY,Junior,1-3,Python; Django; Flask; REST APIs; SQL; Git,Design and develop backend services; Write clean maintainable code; Collaborate with frontend developers; **Participate in code reviews;** Debug and fix production issues,Docker; AWS; PostgreSQL; Redis; Celery,Bachelor's in CS or related,Remote
1,2,Senior Python Developer,INFORMATION-TECHNOLOGY,Senior,5+,Python; Django; Flask; FastAPI; PostgreSQL; Docker; Kubernetes; AWS; **CI/CD;** System Design,Lead backend architecture decisions; Mentor junior developers; Design scalable microservices; Optimize database performance; Implement security best practices; Conduct technical interviews,GraphQL; Kafka; Terraform; Machine Learning,Bachelor's or Master's in CS,San Francisco CA
2,3,Java Developer,INFORMATION-TECHNOLOGY,Junior,1-3,Java; Spring Boot; Hibernate; REST APIs; MySQL; Maven; Git,Develop enterprise applications; Write unit and integration tests; **Participate in agile ceremonies;** Troubleshoot application issues; Document technical specifications,Microservices; Docker; Jenkins; Kafka,Bachelor's in CS or IT,New York NY
3,4,Senior Java Developer,INFORMATION-TECHNOLOGY,Senior,5+,Java; Spring Boot; Spring Cloud; Microservices; Kafka; Kubernetes; AWS; Oracle; **Design Patterns,Architect enterprise solutions;** Lead development teams; Define coding standards; Conduct performance tuning; Integrate third-party systems; Drive technical roadmap,Reactive Programming; GraphQL; Terraform; Machine Learning,Bachelor's or Master's in CS,Austin TX
4,5,Frontend Developer,INFORMATION-TECHNOLOGY,Junior,1-3,JavaScript; HTML5; CSS3; React; Git; Responsive Design,Build responsive user interfaces; Implement designs from Figma; Optimize web performance; **Collaborate with designers;** Write reusable components,TypeScript; Next.js; GraphQL; Testing Library,Bachelor's in CS or Design,Remote


#### There is a few inconsisency in text and the information format in this data. 

In [166]:
import re

In [167]:
# cleaning text fields for better formatting
def clean_text_field(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # adjust markdown bolding
    text = re.sub(r"\*+", "", text)
    # handle spaces around semicolons
    text = re.sub(r"\s*;\s*", "; ", text)
    return text.strip()

In [168]:
# handling list fields that is semicolon separated
def format_list_field(text: str, delimiter: str = ", ") -> str:
    cleaned = clean_text_field(text)
    items = [item.strip() for item in cleaned.split(";") if item.strip()]
    return delimiter.join(items)

In [169]:
# making full processed description
def build_full_processed_description(row: pd.Series) -> str:
    title = row.get("Title", "")
    category = row.get("Category", "")
    exp_level = row.get("ExperienceLevel", "")
    years_exp = row.get("YearsOfExperience", "")
    skills = row.get("Skills", "")
    pref_skills = row.get("PreferredSkills", "")
    responsibilities = row.get("Responsibilities", "")
    education = row.get("Education", "")
    # location = row.get("Location", "") It provide no value in matching.

    full_text = f"""
        Job Title: {title}
        Category: {category}
        Experience Level: {exp_level} ({years_exp} years)
       
        Education Required: {education}

        Required Skills:
        {skills}

        Preferred Skills:
        {pref_skills}

        Key Responsibilities:
        {responsibilities}
    """.strip()

    return full_text

In [170]:
processed_df = df.copy()

# clean individual list and text fields
processed_df["Skills"] = processed_df["Skills"].apply(
    lambda x: format_list_field(x, delimiter=", ")
)
processed_df["PreferredSkills"] = processed_df["PreferredSkills"].apply(
    lambda x: format_list_field(x, delimiter=", ")
)
processed_df["Responsibilities"] = processed_df["Responsibilities"].apply(
    lambda x: format_list_field(x, delimiter=". ")
)

# build full composite text description for vectorization & matching
processed_df["FullProcessedDescription"] = processed_df.apply(
    build_full_processed_description, axis=1
)

In [171]:
processed_df.head(2)

,JobID,Title,Category,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,PreferredSkills,Education,Location,FullProcessedDescription
0,1,Python Developer,INFORMATION-TECHNOLOGY,Junior,1-3,"Python, Django, Flask, REST APIs, SQL, Git",Design and develop backend services. Write clean maintainable code. Collaborate with frontend developers. Participate in code reviews. Debug and fix production issues,"Docker, AWS, PostgreSQL, Redis, Celery",Bachelor's in CS or related,Remote,Job Title: Python Developer\n Category: INFORMATION-TECHNOLOGY\n Experience Level: Junior (1-3 years)\n\n Education Required: Bachelor's in CS or related\n\n Required S...
1,2,Senior Python Developer,INFORMATION-TECHNOLOGY,Senior,5+,"Python, Django, Flask, FastAPI, PostgreSQL, Docker, Kubernetes, AWS, CI/CD, System Design",Lead backend architecture decisions. Mentor junior developers. Design scalable microservices. Optimize database performance. Implement security best practices. Conduct technical interviews,"GraphQL, Kafka, Terraform, Machine Learning",Bachelor's or Master's in CS,San Francisco CA,Job Title: Senior Python Developer\n Category: INFORMATION-TECHNOLOGY\n Experience Level: Senior (5+ years)\n\n Education Required: Bachelor's or Master's in CS\n\n Req...


In [172]:
processed_df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    42
ENGINEERING               18
DIGITAL-MEDIA             10
DATA-SCIENCE              10
FINANCE                    6
HUMAN-RESOURCES            5
MARKETING                  2
Name: count, dtype: int64

#### Now save processed job description file

In [173]:
output_dir = Path("processed")
output_dir.mkdir(parents=True, exist_ok=True)

In [174]:
cleaned_csv_path = output_dir / "final_job_descriptions.csv"
processed_df.to_csv(cleaned_csv_path, index=False)

In [175]:
processed_df["Skills"]

0                                                                               Python, Django, Flask, REST APIs, SQL, Git
1                                Python, Django, Flask, FastAPI, PostgreSQL, Docker, Kubernetes, AWS, CI/CD, System Design
2                                                               Java, Spring Boot, Hibernate, REST APIs, MySQL, Maven, Git
3                          Java, Spring Boot, Spring Cloud, Microservices, Kafka, Kubernetes, AWS, Oracle, Design Patterns
4                                                                   JavaScript, HTML5, CSS3, React, Git, Responsive Design
                                                              ...                                                         
88                       Accounting, GAAP, Excel, QuickBooks, Reconciliation, AP/AR, Financial Statements, Month-End Close
89          Accounting, GAAP, Excel, ERP Systems, Consolidation, Financial Reporting, Team Leadership, Process Improvement
90    Financial 

In [176]:
# 3 job descriptions
list_of3_jobs = processed_df["FullProcessedDescription"].head(1).tolist()
list_of3_jobs

["Job Title: Python Developer\n        Category: INFORMATION-TECHNOLOGY\n        Experience Level: Junior (1-3 years)\n\n        Education Required: Bachelor's in CS or related\n\n        Required Skills:\n        Python, Django, Flask, REST APIs, SQL, Git\n\n        Preferred Skills:\n        Docker, AWS, PostgreSQL, Redis, Celery\n\n        Key Responsibilities:\n        Design and develop backend services. Write clean maintainable code. Collaborate with frontend developers. Participate in code reviews. Debug and fix production issues"]

#### Category wise list of jobs description

In [177]:
processed_df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    42
ENGINEERING               18
DIGITAL-MEDIA             10
DATA-SCIENCE              10
FINANCE                    6
HUMAN-RESOURCES            5
MARKETING                  2
Name: count, dtype: int64

In [178]:
processed_df.head(2)

,JobID,Title,Category,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,PreferredSkills,Education,Location,FullProcessedDescription
0,1,Python Developer,INFORMATION-TECHNOLOGY,Junior,1-3,"Python, Django, Flask, REST APIs, SQL, Git",Design and develop backend services. Write clean maintainable code. Collaborate with frontend developers. Participate in code reviews. Debug and fix production issues,"Docker, AWS, PostgreSQL, Redis, Celery",Bachelor's in CS or related,Remote,Job Title: Python Developer\n Category: INFORMATION-TECHNOLOGY\n Experience Level: Junior (1-3 years)\n\n Education Required: Bachelor's in CS or related\n\n Required S...
1,2,Senior Python Developer,INFORMATION-TECHNOLOGY,Senior,5+,"Python, Django, Flask, FastAPI, PostgreSQL, Docker, Kubernetes, AWS, CI/CD, System Design",Lead backend architecture decisions. Mentor junior developers. Design scalable microservices. Optimize database performance. Implement security best practices. Conduct technical interviews,"GraphQL, Kafka, Terraform, Machine Learning",Bachelor's or Master's in CS,San Francisco CA,Job Title: Senior Python Developer\n Category: INFORMATION-TECHNOLOGY\n Experience Level: Senior (5+ years)\n\n Education Required: Bachelor's or Master's in CS\n\n Req...


### Create Sample JobDescription Data

In [179]:
# pick first 3 of Data Science and IT. 
cats = ["DATA-SCIENCE", "INFORMATION-TECHNOLOGY"]
parts = []
for c in cats:
    parts.append(processed_df[processed_df["Category"].fillna("").str.upper() == c].head(3))

sample_job_descriptions = pd.concat(parts, ignore_index=True)
sample_job_descriptions

,JobID,Title,Category,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,PreferredSkills,Education,Location,FullProcessedDescription
0,61,Machine Learning Researcher,DATA-SCIENCE,Senior,5+,"Python, PyTorch, TensorFlow, GenAI, LLMs, Mathematics, AWS, CI/CD, OpenCV, RAG, Data Validation",Conduct ML research. Publish papers. Develop novel algorithms. Collaborate with engineers. Present at conferences. Mentor junior researchers,"NLP, Computer Vision, Reinforcement Learning, PhD",PhD in CS or related,San Francisco CA,Job Title: Machine Learning Researcher\n Category: DATA-SCIENCE\n Experience Level: Senior (5+ years)\n\n Education Required: PhD in CS or related\n\n Required Skills:\...
1,62,Senior Data Analyst,DATA-SCIENCE,Senior,5+,"SQL, Python, R, Git, Power BI, Statistical Modeling, Data Validation, Team Leadership",Lead analytics projects. Mentor analysts. Design experiments. Build predictive models. Present to executives. Define metrics,"Machine Learning, Airflow, dbt, Domain Expertise",Bachelor's or Master's in Statistics or related,New York NY,Job Title: Senior Data Analyst\n Category: DATA-SCIENCE\n Experience Level: Senior (5+ years)\n\n Education Required: Bachelor's or Master's in Statistics or related\n\n ...
2,63,Business Intelligence Analyst,DATA-SCIENCE,Experienced,3-5,"SQL, Power BI, Git, Data Modeling, ETL, dbt, MLflow",Build BI dashboards. Design data models. Gather requirements. Optimize queries. Train users. Document solutions,"Snowflake, dbt, Looker, Data Governance",Bachelor's in Business or Analytics,Chicago IL,Job Title: Business Intelligence Analyst\n Category: DATA-SCIENCE\n Experience Level: Experienced (3-5 years)\n\n Education Required: Bachelor's in Business or Analytics\n\n ...
3,1,Python Developer,INFORMATION-TECHNOLOGY,Junior,1-3,"Python, Django, Flask, REST APIs, SQL, Git",Design and develop backend services. Write clean maintainable code. Collaborate with frontend developers. Participate in code reviews. Debug and fix production issues,"Docker, AWS, PostgreSQL, Redis, Celery",Bachelor's in CS or related,Remote,Job Title: Python Developer\n Category: INFORMATION-TECHNOLOGY\n Experience Level: Junior (1-3 years)\n\n Education Required: Bachelor's in CS or related\n\n Required S...
4,2,Senior Python Developer,INFORMATION-TECHNOLOGY,Senior,5+,"Python, Django, Flask, FastAPI, PostgreSQL, Docker, Kubernetes, AWS, CI/CD, System Design",Lead backend architecture decisions. Mentor junior developers. Design scalable microservices. Optimize database performance. Implement security best practices. Conduct technical interviews,"GraphQL, Kafka, Terraform, Machine Learning",Bachelor's or Master's in CS,San Francisco CA,Job Title: Senior Python Developer\n Category: INFORMATION-TECHNOLOGY\n Experience Level: Senior (5+ years)\n\n Education Required: Bachelor's or Master's in CS\n\n Req...
5,3,Java Developer,INFORMATION-TECHNOLOGY,Junior,1-3,"Java, Spring Boot, Hibernate, REST APIs, MySQL, Maven, Git",Develop enterprise applications. Write unit and integration tests. Participate in agile ceremonies. Troubleshoot application issues. Document technical specifications,"Microservices, Docker, Jenkins, Kafka",Bachelor's in CS or IT,New York NY,Job Title: Java Developer\n Category: INFORMATION-TECHNOLOGY\n Experience Level: Junior (1-3 years)\n\n Education Required: Bachelor's in CS or IT\n\n Required Skills:\...


In [180]:
# save to processed/processed_sample_jobs/sample_job_descriptions.csv
out_dir = Path("processed/processed_sample_jobs/")
out_dir.mkdir(parents=True, exist_ok=True)
sample_path = out_dir / "sample_job_descriptions.csv"
sample_job_descriptions.to_csv(sample_path, index=False)